In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

np.random.seed(42)

plt.rcParams.update({
    'figure.dpi'       : 120,
    'axes.spines.top'  : False,
    'axes.spines.right': False,
    'font.size'        : 12,
})
print('All imports OK ✓')

In [ ]:
from sklearn.datasets import load_breast_cancer
import pandas as pd

cancer_data = load_breast_cancer()

df = pd.DataFrame(cancer_data.data, columns=cancer_data.feature_names)

df['target'] = cancer_data.target

print(df.head())

In [ ]:
load_breast_cancer().feature_names


In [ ]:
print('=== Shape ===')
print(df.shape)

print('\n=== Data types ===')
print(df.dtypes)

print('\n=== Missing values per column ===')
print(df.isnull().sum())

print('\n=== Summary statistics ===')
df.describe().round(3)

In [ ]:
import matplotlib.pyplot as plt
import math

# Calculate how many rows you need for 3 columns per row
num_cols = len(df.columns)
cols_per_row = 3
num_rows = math.ceil(num_cols / cols_per_row)

# Create the figure with the dynamic grid
fig, axes = plt.subplots(num_rows, cols_per_row, figsize=(15, num_rows * 4))
axes = axes.flatten()

for i, col in enumerate(df.columns):
    axes[i].hist(df[col], bins=40, color='steelblue', edgecolor='white', linewidth=0.3)
    axes[i].set_title(col, fontsize=10)
    axes[i].set_xlabel('')

# Hide any extra empty subplots if the grid is larger than the number of columns
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

In [ ]:
X = cancer_data.data   # shape (n, d)
y = cancer_data.target  # shape (n,)
print(cancer_data.target_names)
feature_names = cancer_data.feature_names

print(f'X shape : {X.shape}   ({X.shape[0]} samples, {X.shape[1]} features)')
print(f'y shape : {y.shape}')
print(f'Feature names: {feature_names}')
print(f'Target (Diagnosis) — min: {y.min():.2f},  max: {y.max():.2f},  mean: {y.mean():.2f}')

In [ ]:
n = len(X)
n_train = int(0.8 * n)

# Shuffle indices first — order in the dataset may not be random!
idx = np.arange(n)
rng = np.random.default_rng(42)
rng.shuffle(idx)

train_idx = idx[:n_train]
test_idx  = idx[n_train:]

X_train_manual = X[train_idx]
y_train_manual = y[train_idx]
X_test_manual  = X[test_idx]
y_test_manual  = y[test_idx]

print(f'Total samples : {n}')
print(f'Train samples : {len(X_train_manual)}  ({100*len(X_train_manual)/n:.1f}%)')
print(f'Test  samples : {len(X_test_manual)}   ({100*len(X_test_manual)/n:.1f}%)')
print(f'\nTrain mean y  : {y_train_manual.mean():.4f}')
print(f'Test  mean y  : {y_test_manual.mean():.4f}   ← should be similar to train')

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size    = 0.20,    # 20% held out
    random_state = 42,      # reproducible shuffle
)

print(f'X_train : {X_train.shape}   y_train : {y_train.shape}')
print(f'X_test  : {X_test.shape}    y_test  : {y_test.shape}')

In [ ]:
scaler = StandardScaler()

# Fit ONLY on training data — never on test!
scaler.fit(X_train)

# Transform both sets using the training statistics
X_train_scaled = scaler.transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print('Before scaling — MedInc column stats:')
print(f'  train mean={X_train[:,0].mean():.3f}, std={X_train[:,0].std():.3f}')

print('\nAfter scaling — MedInc column stats:')
print(f'  train mean={X_train_scaled[:,0].mean():.6f}  (≈ 0)')
print(f'  train std ={X_train_scaled[:,0].std():.6f}   (≈ 1)')
print(f'  test  mean={X_test_scaled[:,0].mean():.4f}   (close but not exactly 0 — that is OK)')

In [ ]:
# Visualise: before vs after scaling for all 8 features
fig, axes = plt.subplots(2, 8, figsize=(18, 5))

for i in range(8):
    axes[0, i].hist(X_train[:, i],        bins=30, color='steelblue', edgecolor='w', lw=0.2)
    axes[1, i].hist(X_train_scaled[:, i], bins=30, color='crimson',   edgecolor='w', lw=0.2)
    axes[0, i].set_title(feature_names[i], fontsize=8)
    axes[1, i].set_title('scaled', fontsize=8)

axes[0, 0].set_ylabel('Before scaling')
axes[1, 0].set_ylabel('After scaling')
plt.suptitle('Feature distributions before and after standardisation', fontsize=12)
plt.tight_layout()
plt.show()